#`mountUmount(`<font size="3px" color="#01c968">`Gdrive`</font>`)`



In [ ]:
#@markdown <br><center><img src='https://upload.wikimedia.org/wikipedia/commons/thumb/d/da/Google_Drive_logo.png/600px-Google_Drive_logo.png' height="50" alt="Gdrive-logo"/></center>
#@markdown <center><h3>Mount Gdrive to /content/drive</h3></center><br>
MODE = "MOUNT" #@param ["MOUNT", "UNMOUNT"]
#Mount your Gdrive!
from google.colab import drive
drive.mount._DEBUG = False
if MODE == "MOUNT":
  drive.mount('/content/drive', force_remount=True)
elif MODE == "UNMOUNT":
  try:
    drive.flush_and_unmount()
  except ValueError:
    pass
  get_ipython().system_raw("rm -rf /root/.config/Google/DriveFS")

#`Setup And Update ComfyUI`



In [ ]:
from pathlib import Path

OPTIONS = {}

DRIVE_PATH = ""  # @param {type:"string"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
WORKSPACE = '/content/ComfyUI'
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI

if DRIVE_PATH:

    WORKSPACE = DRIVE_PATH+"/ComfyUI"
    %cd {DRIVE_PATH}

![ ! -d WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-
  !git pull

!echo -= Install dependencies =-
!pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121 --extra-index-url https://download.pytorch.org/whl/cu118 --extra-index-url https://download.pytorch.org/whl/cu117

%cd {WORKSPACE}

# `Models Download`

## Download MODELS

In [ ]:
%cd {WORKSPACE}
# Install Aria2
!apt-get -y install -qq aria2


from google.colab import userdata

CIVITAI_API_TOKEN = userdata.get('CIVITAI_API_TOKEN')
print("Loaded API key:", "✅" if CIVITAI_API_TOKEN else "❌ Not found")

# Install custom nodes

def install_custom_node(url):
  %cd /content/ComfyUI/custom_nodes
  !git clone {url}

def downloadModel(url, filename = None):
  if 'huggingface.co' in url:
    if filename is None:
      filename = url.split('/')[-1]
      filename = filename.removesuffix('?download=true')
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {url}  -o {filename}
  else:
    # civitai
    if filename:
      !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {url}?token={CIVITAI_API_TOKEN} -o {filename}
    else:
      !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {url}?token={CIVITAI_API_TOKEN}


# install custom nodes
#install_custom_node('https://github.com/ltdrdata/ComfyUI-Manager.git')
#install_custom_node('https://github.com/ltdrdata/ComfyUI-Impact-Pack')


# download models
%cd ./models/checkpoints

# CyberRealistic Pony
downloadModel('https://civitai.com/api/download/models/2071650')

# Protovision XL
#downloadModel('https://civitai.com/api/download/models/201514')
# Juggernaut XL Hyper
#downloadModel('https://civitai.com/api/download/models/471120')
# Juppernaut X
#downloadModel('https://civitai.com/api/download/models/456194')
# Juggernaut v8
#downloadModel('https://civitai.com/api/download/models/288982')

# Huggingface example
#downloadModel('https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix2/AbyssOrangeMix2_hard.safetensors?download=true')

%cd {WORKSPACE}

## OTHERS

In [ ]:
# SD1.5
!wget -c https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt -P ./models/checkpoints/

# Some SD1.5 anime style
!wget -c https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix2/AbyssOrangeMix2_hard.safetensors -P ./models/checkpoints/

# VAE
!wget -c https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors -P ./models/vae/

## LIST MODELS

In [3]:
!ls -al ./models/checkpoints/

total 6775444
drwxr-xr-x  2 root root       4096 Aug 30 23:40 .
drwxr-xr-x 22 root root       4096 Aug 30 23:39 ..
-rw-r--r--  1 root root 6938040682 Aug 30 23:40 cyberrealisticPony_v127Alt.safetensors
-rw-r--r--  1 root root          0 Aug 30 23:39 put_checkpoints_here


# `START ComfyUI  & Expose Server (MANUAL)`

## Install cloudflared

In [ ]:
# Cài cloudflared từ repo chính thức của Cloudflare (Debian/Ubuntu)
import os

SUDO = "" if os.geteuid() == 0 else "sudo"

!{SUDO} mkdir -p --mode=0755 /usr/share/keyrings
!curl -fsSL https://pkg.cloudflare.com/cloudflare-main.gpg | {SUDO} tee /usr/share/keyrings/cloudflare-main.gpg >/dev/null
!echo 'deb [signed-by=/usr/share/keyrings/cloudflare-main.gpg] https://pkg.cloudflare.com/cloudflared any main' | {SUDO} tee /etc/apt/sources.list.d/cloudflared.list
!{SUDO} apt-get update && {SUDO} apt-get install -y cloudflared
!cloudflared --version


In [ ]:
!python main.py --dont-print-server

# `START ComfyUI & Expose Server`

## Install cloudflared

In [ ]:
# Cài cloudflared từ repo chính thức của Cloudflare (Debian/Ubuntu)
import os

SUDO = "" if os.geteuid() == 0 else "sudo"

!{SUDO} mkdir -p --mode=0755 /usr/share/keyrings
!curl -fsSL https://pkg.cloudflare.com/cloudflare-main.gpg | {SUDO} tee /usr/share/keyrings/cloudflare-main.gpg >/dev/null
!echo 'deb [signed-by=/usr/share/keyrings/cloudflare-main.gpg] https://pkg.cloudflare.com/cloudflared any main' | {SUDO} tee /etc/apt/sources.list.d/cloudflared.list
!{SUDO} apt-get update && {SUDO} apt-get install -y cloudflared
!cloudflared --version


## CF Tunnel

In [ ]:
# Cloudflare named tunnel (không dùng quick tunnel / trycloudflare.com)
import subprocess
import threading
import time
import socket

from google.colab import userdata

CLOUDFLARE_TUNNEL_TOKEN = userdata.get('CLOUDFLARE_TUNNEL_TOKEN')
if not CLOUDFLARE_TUNNEL_TOKEN:
  raise ValueError(
      "Thiếu Colab secret 'CLOUDFLARE_TUNNEL_TOKEN'. Trong Cloudflare Zero Trust -> Networks -> "
      "Tunnels -> <tunnel của bạn> -> 'Install and run a connector' -> copy token."
  )

TUNNEL_HOSTNAME = ""  # @param {type:"string"}  # hostname public đã map trong Cloudflare, vd: comfy.example.com
PORT = 8188


def wait_port(port, timeout=600):
  deadline = time.time() + timeout
  while time.time() < deadline:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
      sock.settimeout(1)
      if sock.connect_ex(('127.0.0.1', port)) == 0:
        return True
    time.sleep(0.5)
  return False


def tunnel_thread(port):
  if not wait_port(port):
    print("\nComfyUI chưa sẵn sàng sau 10 phút - tunnel không được khởi động.\n")
    return
  print("\nComfyUI đã sẵn sàng, khởi động Cloudflare Tunnel (named tunnel)...\n")
  p = subprocess.Popen(
      ["cloudflared", "tunnel", "--no-autoupdate", "run", "--token", CLOUDFLARE_TUNNEL_TOKEN],
      stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
  for line in p.stdout:
    l = line.decode(errors='replace')
    if "Registered tunnel connection" in l:
      print("Tunnel đã đăng ký. Mở ComfyUI tại: https://" +
            (TUNNEL_HOSTNAME or "<hostname đã cấu hình trong Cloudflare>"))
    print(l, end='')


threading.Thread(target=tunnel_thread, daemon=True, args=(PORT,)).start()

%cd {WORKSPACE}
!python main.py --dont-print-server
